In [1]:
!pip install transformers datasets peft accelerate bitsandbytes wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 105.1 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

In [4]:
import wandb
wandb.login(secret_value_0)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhalaniakshat (bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
os.environ['NCCL_P2P_DISABLE']='1'
os.environ['NCCL_IB_DISABLE']='1'

In [6]:
import random,torch
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,TrainingArguments,BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

In [7]:
da=load_dataset('Crownelius/Opus-4.6-Reasoning-3300x',split='train')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.74M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2160 [00:00<?, ? examples/s]

In [8]:
def cott(dn):
    if random.random()<0.6:
        t=f'''###Instruction: Solve step by step.
### Problem:{dn['problem']}
### Thinking:{dn['thinking']}
### Final Answer:{dn['solution']}'''
    else:
        t=f'''###Instruction: Solve the problem.
###Problem:{dn['problem']}
###Final Answer: {dn['solution']}
'''
    return {'text':t}

In [9]:
da=da.map(cott)

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

In [10]:
tok=AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct',trust_remote_code=True,add_eos_token=True,use_fast=True)
tok.padding_side = "right"
tok.pad_token = tok.eos_token

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [11]:
bn=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)

In [12]:
dev={'':0}

In [13]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=True
)

config.rope_scaling = None

In [14]:
mod=AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct',config=config,quantization_config=bn,device_map='auto',torch_dtype="auto",trust_remote_code=True)
mod.config.pad_token_id=tok.pad_token_id

modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [15]:
lor=LoraConfig(r=16,lora_alpha=32,target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],lora_dropout=0.05,bias="none",task_type="CAUSAL_LM")

In [16]:
mod=get_peft_model(mod,lor)
mod.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [17]:
def toke(ds):
    return tok(ds['text'],truncation=True,padding='max_length',max_length=768)
da=da.map(toke,batched=True)

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

In [18]:
da=da.remove_columns([col for col in da.column_names if col not in ["input_ids", "attention_mask"]])

In [19]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tok,
    mlm=False
)

In [20]:
trags=TrainingArguments(output_dir='./phiopus',
per_device_train_batch_size=1,gradient_accumulation_steps=16,
                       num_train_epochs=4,learning_rate=1e-4,fp16=True,logging_steps=10,save_steps=200,report_to='wandb')

In [24]:
from transformers import Trainer

class CustomTrainer(Trainer):
    def log(self, logs, *args, **kwargs):
        super().log(logs, *args, **kwargs)
        if "loss" in logs:
            print(f"Step {self.state.global_step}: Loss = {logs['loss']:.4f}")

In [22]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
trainer = CustomTrainer(
    model=mod,
    args=trags,
    train_dataset=da,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
10,10.154121


Step 10: Loss = 10.1541
